# Pure DMPC vs DMPC-RL — Comparison Analysis

Head-to-head comparison of:
- **Pure DMPC** — fixed baseline Q/R cost matrices, no learning.
- **DMPC-RL** — MAPPO-adaptive Q/R scale vectors; trained via
  `scripts/train_mappo.py`.

across all three ISR task scenarios:
1. `area_surveillance` — wide-area persistent coverage
2. `threat_response`   — rapid hostile target detection & tracking
3. `search_and_track`  — locate & track mobile targets

**Workflow:**
```bash
# 1. Train the MAPPO policy (once)
python scripts/train_mappo.py

# 2. Run both methods for each scenario
for sc in area_surveillance threat_response search_and_track; do
  python scripts/run_dmpc.py    --scenario $sc --episodes 10
  python scripts/run_dmpc_rl.py --scenario $sc --episodes 10
done

# 3. Open this notebook and run all cells
```

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

plt.rcParams.update({'figure.figsize': (14, 6), 'font.size': 11})

RESULTS_ROOT = Path('../data/results')
SCENARIOS    = ['area_surveillance', 'threat_response', 'search_and_track']
METHODS      = {
    'Pure DMPC': RESULTS_ROOT / 'dmpc',
    'DMPC-RL':   RESULTS_ROOT / 'dmpc_rl',
}
COLORS = {'Pure DMPC': 'steelblue', 'DMPC-RL': 'tomato'}

def load_all(method_dirs: dict) -> dict[str, dict[str, list[dict]]]:
    """Return {method: {scenario: [episode_dicts]}}."""
    out: dict[str, dict[str, list[dict]]] = {}
    for method, base_dir in method_dirs.items():
        sc_eps: dict[str, list[dict]] = {s: [] for s in SCENARIOS}
        if base_dir.exists():
            for path in sorted(base_dir.glob('*.json')):
                with open(path) as f:
                    episodes = json.load(f)
                for ep in (episodes if isinstance(episodes, list) else [episodes]):
                    sc = ep.get('scenario', path.stem.split('_2')[0])
                    if sc in sc_eps:
                        sc_eps[sc].append(ep)
        total = sum(len(v) for v in sc_eps.values())
        print(f'{method}: {total} episodes')
        out[method] = sc_eps
    return out

data = load_all(METHODS)

def get_metric(data, method, scenario, key):
    eps = data.get(method, {}).get(scenario, [])
    return np.array([ep.get(key, np.nan) for ep in eps], dtype=float)

## 1 · Total Reward

Higher reward indicates better task performance.  DMPC-RL is expected to
outperform Pure DMPC once the policy has been trained for sufficient
timesteps.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False)
for ax, sc in zip(axes, SCENARIOS):
    for i, (method, color) in enumerate(COLORS.items()):
        r = get_metric(data, method, sc, 'total_reward')
        r = r[~np.isnan(r)]
        if len(r) == 0:
            continue
        x = np.full(len(r), i) + np.random.uniform(-0.1, 0.1, len(r))
        ax.scatter(x, r, color=color, alpha=0.6, s=40, zorder=3)
        ax.errorbar(i, np.mean(r), yerr=np.std(r), fmt='D', color=color,
                    markersize=8, capsize=6, lw=2, zorder=4)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(list(COLORS.keys()), fontsize=10)
    ax.set_title(sc.replace('_', ' ').title(), fontweight='bold')
    ax.set_ylabel('Total reward')
    ax.grid(axis='y', alpha=0.3)
fig.suptitle('Total Reward: Pure DMPC vs DMPC-RL', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('comparison_reward.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved comparison_reward.png')

## 2 · DMPC Solve Time

DMPC-RL adds an inference pass before each DMPC solve.  The extra
overhead should remain well below the 20 ms control period (50 Hz).

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(SCENARIOS))
w = 0.35
for k, (method, color) in enumerate(COLORS.items()):
    means = [np.nanmean(get_metric(data, method, sc, 'mean_solve_time_ms')) for sc in SCENARIOS]
    stds  = [np.nanstd( get_metric(data, method, sc, 'mean_solve_time_ms')) for sc in SCENARIOS]
    ax.bar(x + (k - 0.5) * w, means, w, yerr=stds,
           label=method, color=color, capsize=4, alpha=0.85)
ax.axhline(20, color='grey', ls='--', lw=1.5, label='50 Hz deadline (20 ms)')
ax.set_xticks(x)
ax.set_xticklabels([s.replace('_', '\n') for s in SCENARIOS])
ax.set_ylabel('Mean solve time (ms)')
ax.set_title('DMPC Solve Time per Scenario', fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 3 · Battery Efficiency

Remaining battery fraction at episode end (1.0 = full, 0.0 = depleted).
DMPC-RL should learn to reduce unnecessary control effort and preserve
battery life where the mission allows it.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(SCENARIOS))
w = 0.35
for k, (method, color) in enumerate(COLORS.items()):
    means = [np.nanmean(get_metric(data, method, sc, 'final_battery_mean')) for sc in SCENARIOS]
    stds  = [np.nanstd( get_metric(data, method, sc, 'final_battery_mean')) for sc in SCENARIOS]
    ax.bar(x + (k - 0.5) * w, means, w, yerr=stds,
           label=method, color=color, capsize=4, alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels([s.replace('_', '\n') for s in SCENARIOS])
ax.set_ylabel('Final battery (0–1)')
ax.set_title('Battery Level at Episode End', fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)
ax.set_ylim([0, 1.1])
plt.tight_layout()
plt.show()

## 4 · DMPC-RL Adaptation Activity

Mean Q-scale output by the MAPPO policy.  A value of 1.0 means the RL
policy is not changing the cost matrix.  Deviations reveal how actively
the policy adapts the DMPC weights to each scenario.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(SCENARIOS))
w = 0.5
means_q = [np.nanmean(get_metric(data, 'DMPC-RL', sc, 'mean_q_scale')) for sc in SCENARIOS]
stds_q  = [np.nanstd( get_metric(data, 'DMPC-RL', sc, 'mean_q_scale')) for sc in SCENARIOS]
std_dev  = [np.nanmean(get_metric(data, 'DMPC-RL', sc, 'std_q_scale'))  for sc in SCENARIOS]

ax.bar(x, means_q, w, yerr=stds_q, color='tomato', capsize=4, alpha=0.85, label='Mean Q-scale')
ax.errorbar(x, means_q, yerr=std_dev, fmt='none', color='black', capsize=4, label='Within-ep std')
ax.axhline(1.0, color='steelblue', ls='--', lw=1.5, label='Pure DMPC baseline (1.0)')
ax.set_xticks(x)
ax.set_xticklabels([s.replace('_', '\n') for s in SCENARIOS])
ax.set_ylabel('Mean Q-scale')
ax.set_title('DMPC-RL Adaptation Activity (Q-scale)', fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 5 · Statistical Significance

Welch's t-test comparing per-episode rewards between Pure DMPC and
DMPC-RL for each scenario.

In [ ]:
from scipy import stats

print(f'{'Scenario':<22} {'DMPC μ':>10} {'RL μ':>10} {'Δ μ':>10} {'p-value':>10} {'sig':>5}')
print('-' * 65)
for sc in SCENARIOS:
    d  = get_metric(data, 'Pure DMPC', sc, 'total_reward')
    rl = get_metric(data, 'DMPC-RL',   sc, 'total_reward')
    d, rl = d[~np.isnan(d)], rl[~np.isnan(rl)]
    if len(d) < 2 or len(rl) < 2:
        print(f'{sc:<22}  (insufficient data — run more episodes)')
        continue
    t_stat, p_val = stats.ttest_ind(d, rl, equal_var=False)
    delta = np.mean(rl) - np.mean(d)
    sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'
    print(f'{sc:<22} {np.mean(d):>10.2f} {np.mean(rl):>10.2f} {delta:>+10.2f} {p_val:>10.4f} {sig:>5}')

## 6 · Radar / Spider Chart — Multi-Metric Summary

Normalised radar chart comparing Pure DMPC and DMPC-RL across
reward, battery, and (inverse) solve-time metrics.

In [ ]:
from matplotlib.patches import FancyArrowPatch

METRICS = {
    'Reward':       ('total_reward',       1,   None),   # higher is better
    'Battery':      ('final_battery_mean', 1,   None),
    'Speed\n(1/ms)': ('mean_solve_time_ms', -1,  None),   # lower is better → invert
}

def normalise(arr, higher_better=True, global_max=None, global_min=None):
    """Scale to [0,1]; flip if lower_is_better."""
    arr = np.asarray(arr, dtype=float)
    vmin = global_min if global_min is not None else np.nanmin(arr)
    vmax = global_max if global_max is not None else np.nanmax(arr)
    if vmax == vmin:
        return np.ones_like(arr) * 0.5
    norm = (arr - vmin) / (vmax - vmin)
    return norm if higher_better else 1 - norm

n_metrics = len(METRICS)
angles = np.linspace(0, 2 * np.pi, n_metrics, endpoint=False).tolist()
angles += angles[:1]  # close polygon

fig, axes = plt.subplots(1, len(SCENARIOS), figsize=(16, 5),
                         subplot_kw={'polar': True})

for ax, sc in zip(axes, SCENARIOS):
    for method, color in COLORS.items():
        vals = []
        for label, (key, sign, _) in METRICS.items():
            v = get_metric(data, method, sc, key)
            v = v[~np.isnan(v)]
            mean_v = np.nanmean(v) if len(v) > 0 else np.nan
            vals.append(mean_v if sign > 0 else -mean_v if not np.isnan(mean_v) else np.nan)
        # Simple 0-1 normalisation within each radar plot
        vals_arr = np.array(vals, dtype=float)
        vmin, vmax = np.nanmin(vals_arr), np.nanmax(vals_arr)
        if vmax > vmin:
            vals_norm = (vals_arr - vmin) / (vmax - vmin)
        else:
            vals_norm = np.zeros_like(vals_arr)
        vals_norm = np.nan_to_num(vals_norm, nan=0.0).tolist()
        vals_norm += vals_norm[:1]
        ax.plot(angles, vals_norm, color=color, lw=2, label=method)
        ax.fill(angles, vals_norm, color=color, alpha=0.15)
    ax.set_thetagrids(np.degrees(angles[:-1]), list(METRICS.keys()), fontsize=9)
    ax.set_ylim(0, 1)
    ax.set_title(sc.replace('_', '\n'), fontweight='bold', pad=14)
    ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=8)

fig.suptitle('Multi-Metric Radar: Pure DMPC vs DMPC-RL', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('comparison_radar.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved comparison_radar.png')